# 02 · Detection on the 17 base models (§5.3–5.4): Table 3

This notebook recomputes the localization results of the Gram-neighbor detector on the official base checkpoints.

| Paper item | What is computed |
|---|---|
| Table 3 | $\ell^*$, Exact %, MAE, $\hat\ell_0$ (unedited control) and lift for the 17 models |
| §5.4, Abstract, Conclusion | 13,608 of 17,000 edits localized (80.0%) |
| §5.4 *Clear detections* | eight models with at least 98.5% and a control on another layer, lift 2.81–15.61 |
| §5.4 *False positives* | three models whose unedited checkpoint already selects $\ell^*$; 10,608 of 14,000 without them |
| §5.4 *Failures* | Granite 4.1, Granite 4.0 Micro, Olmo 3: natural peak 1.65–2.24× higher, lift 1.02–1.25 |
| §5.4 *Where misses land* | 3,388 of 3,392 misses return $\hat\ell_0$; four DeepSeek misses return layer 5 |
| §5.7 | 11 models detected without flagging the unedited checkpoint, at 73.3–99.5% |

**Definitions used below.**
*Exact* is the share of the 1,000 edits for which the detector returns the edited layer $\ell^*$.
*MAE* is the mean absolute distance between the returned layer and $\ell^*$.
$\hat\ell_0$ is the layer the detector returns on the unedited checkpoint; when $\hat\ell_0 = \ell^*$ the decision rule flags the unedited checkpoint (false positive).
*Lift* is the mean detector score $s_{\ell^*}$ over the edited checkpoints divided by $s_{\ell^*}$ on the unedited checkpoint.

**Inputs:** `existing-renderer/classic-gram/<model>/summary.json` (detected-layer histogram and unedited control),
`data/classic-layer-profile-statistics.csv` (score $s_\ell$ per layer), `data/classic-family-summary.csv`.

In [ ]:
import pandas as pd
from paper_common import (MODELS, EXCLUDED, OUT, read_csv, base_model_results, inputs_used,
                          Checks, latex_rows_equal, diff_rows)

pd.set_option("display.precision", 2)
checks = Checks()

df = base_model_results()
fam = read_csv("data/classic-family-summary.csv").set_index("model")
checks.same("exact counts agree with classic-family-summary.csv", "consistency",
            df.set_index("key").exact.to_dict(), fam.loc[df.key, "gram_exact_count"].astype(int).to_dict())
df[["name", "target", "exact_pct", "mae", "l0", "control_flags", "lift", "detected"]]

## Table 3

$^\dagger$ marks the models whose unedited checkpoint already selects $\ell^*$. The rows are written to `outputs/table3_rows.tex`.

In [ ]:
def table3_rows(d):
    lines = []
    for i, (_, g) in enumerate(d.groupby("period", sort=False)):
        if i:
            lines.append(r"\addlinespace")
        for r in g.itertuples():
            l0 = f"{r.l0}$^\\dagger$" if r.control_flags else f"{r.l0}"
            lines.append(f"{r.name:<22} & {r.target:2d} & {r.exact_pct:5.1f} & {r.mae:.2f} & {l0:>13} & {r.lift:5.2f} \\\\")
    return "\n".join(lines)

PAPER_TABLE3 = r"""
GPT-2 Medium           &  8 &  73.3 & 1.60 &            14 &  4.04 \\
GPT-2 Large            & 12 &  98.5 & 0.17 &            23 &  8.67 \\
GPT-2 XL               & 16 &  98.5 & 0.39 &            42 &  5.29 \\
GPT-J-6B               &  5 &  99.4 & 0.01 &             7 &  3.55 \\
\addlinespace
OPT-6.7B               & 14 &  86.2 & 0.14 &            15 &  2.56 \\
Falcon-7B              &  6 &  86.0 & 0.98 &            13 &  2.63 \\
Llama-2-7B             &  5 & 100.0 & 0.00 &   5$^\dagger$ &  2.32 \\
Mistral-7B v0.1        &  7 & 100.0 & 0.00 &   7$^\dagger$ &  1.49 \\
DeepSeek-LLM-7B Base   &  6 &  99.5 & 0.01 &             8 &  2.81 \\
\addlinespace
Mistral-7B v0.3        &  7 & 100.0 & 0.00 &   7$^\dagger$ &  1.44 \\
Qwen3-4B               &  8 &  99.3 & 0.09 &            21 &  3.64 \\
Qwen3-8B               & 10 &  99.5 & 0.06 &            21 & 15.61 \\
Granite~4.0 Micro      & 10 &  10.4 & 3.58 &             6 &  1.25 \\
Olmo~3 7B              &  6 &   0.3 & 1.99 &             4 &  1.02 \\
Ministral~3 8B         &  7 &  98.9 & 0.01 &             6 &  2.91 \\
\addlinespace
Qwen3.5-4B             &  5 &  99.3 & 0.01 &             6 &  3.84 \\
Granite~4.1 8B         & 13 &  11.7 & 6.18 &            20 &  1.21 \\
"""

rows3 = table3_rows(df)
(OUT / "table3_rows.tex").write_text(rows3 + "\n")
checks.same("Table 3 rows equal the paper", "Table 3", latex_rows_equal(rows3, PAPER_TABLE3), True)
print(rows3)
diff_rows(rows3, PAPER_TABLE3)

## Totals (§5.4, Abstract, Conclusion)

In [ ]:
checks.value("edits localized at l*", "§5.4", df.exact.sum(), 13608)
checks.value("edits in total", "§5.4", df.n.sum(), 17000)
checks.printed("share localized (%)", "§5.4, Conclusion; 80 percent in the Abstract", 100 * df.exact.sum() / df.n.sum(), "80.0")

## Groups of models (§5.4)

*Clear detections*: at least 98.5% exact and an unedited control on another layer.
*False positives*: the unedited control already selects $\ell^*$.
*Failures*: the detector localizes few edits (below 50% here; the three models are far below).

In [ ]:
clear = df[(df.exact_pct >= 98.5) & ~df.control_flags]
fp = df[df.control_flags]
fail = df[(df.exact_pct < 50) & ~df.control_flags]
rest = df[~df.key.isin(set(clear.key) | set(fp.key) | set(fail.key))]

checks.same("clear detections", "§5.4, Abstract, Conclusion", set(clear.name),
            {"DeepSeek-LLM-7B Base", "GPT-J-6B", "GPT-2 Large", "GPT-2 XL", "Ministral~3 8B", "Qwen3-4B", "Qwen3-8B", "Qwen3.5-4B"})
checks.printed("smallest lift among clear detections", "§5.4", clear.lift.min(), "2.81", 2)
checks.printed("largest lift among clear detections", "§5.4", clear.lift.max(), "15.61", 2)
for name, value in [("Falcon-7B", "86.0"), ("OPT-6.7B", "86.2"), ("GPT-2 Medium", "73.3")]:
    checks.printed(f"exact rate of {name}", "§5.4", df.set_index("name").loc[name, "exact_pct"], value)
checks.same("models in no other group", "§5.4", set(rest.name), {"Falcon-7B", "OPT-6.7B", "GPT-2 Medium"})

checks.same("unedited checkpoint flagged", "§5.4, Abstract", set(fp.name), {"Llama-2-7B", "Mistral-7B v0.1", "Mistral-7B v0.3"})
checks.printed("lift on Llama-2-7B", "§5.4", fp.set_index("name").loc["Llama-2-7B", "lift"], "2.32", 2)
mistral = fp[fp.name.str.startswith("Mistral")]
checks.printed("smallest lift on the Mistral models", "§5.4", mistral.lift.min(), "1.44", 2)
checks.printed("largest lift on the Mistral models", "§5.4", mistral.lift.max(), "1.49", 2)
without_fp = df[~df.control_flags]
checks.value("edits localized without the three FP models", "§5.4", without_fp.exact.sum(), 10608)
checks.value("edits without the three FP models", "§5.4", without_fp.n.sum(), 14000)
checks.printed("share without the three FP models (%)", "§5.4", 100 * without_fp.exact.sum() / without_fp.n.sum(), "75.8")

checks.same("failures", "§5.4, Abstract", set(fail.name), {"Granite~4.1 8B", "Granite~4.0 Micro", "Olmo~3 7B"})
for name, value in [("Granite~4.1 8B", "11.7"), ("Granite~4.0 Micro", "10.4"), ("Olmo~3 7B", "0.3")]:
    checks.printed(f"exact rate of {name}", "§5.4", df.set_index("name").loc[name, "exact_pct"], value)
peak_ratio = fail.s_l0_control / fail.s_target_control  # unedited score at its own peak vs. at l*
checks.printed("natural peak / score at l*, smallest", "§5.4 Failures", peak_ratio.min(), "1.65", 2)
checks.printed("natural peak / score at l*, largest", "§5.4 Failures", peak_ratio.max(), "2.24", 2)
checks.printed("lift on the failures, smallest", "§5.4 Failures", fail.lift.min(), "1.02", 2)
checks.printed("lift on the failures, largest", "§5.4 Failures", fail.lift.max(), "1.25", 2)

detected_without_fp = df[~df.control_flags & ~df.key.isin(fail.key)]
checks.value("models detected without flagging the unedited checkpoint", "§5.7", len(detected_without_fp), 11)
checks.printed("their lowest exact rate", "§5.7", detected_without_fp.exact_pct.min(), "73.3")
checks.printed("their highest exact rate", "§5.7", detected_without_fp.exact_pct.max(), "99.5")
pd.DataFrame({"group": ["clear detections", "other detections", "false positives", "failures"],
              "models": [", ".join(g.name) for g in (clear, rest, fp, fail)]})

## Where misses land (§5.4)

3,388 of the 3,392 misses return the unedited control layer. The four exceptions
are DeepSeek edits localized to layer 5 (target 6, unedited control 8).
MAE is computed from actual layer distances. The miss-rate × control-distance
identity holds for the other 16 models, but not for DeepSeek.


In [ ]:
df["misses_on_l0_only"] = [set(d) <= {t, l0} for d, t, l0 in zip(df.detected, df.target, df.l0)]
df["mae_from_rule"] = (1 - df.exact / df.n) * (df.target - df.l0).abs()
checks.value("total misses", "§5.4", int((df.n - df.exact).sum()), 3392)
checks.value("misses returning the unedited control layer", "§5.4",
             sum(d.get(l0, 0) for d, t, l0 in zip(df.detected, df.target, df.l0) if t != l0), 3388)
checks.same("only DeepSeek has misses outside the control layer", "§5.4",
            set(df.loc[~df.misses_on_l0_only, "key"]), {"deepseek-7b-base"})
ds = df.set_index("key").loc["deepseek-7b-base"]
checks.same("DeepSeek misses outside target and control", "§5.4",
            {layer: n for layer, n in ds.detected.items() if layer not in {ds.target, ds.l0}}, {5: 4})
checks.value("DeepSeek MAE uses actual distances", "Table 3", ds.mae, 0.006, 1e-12)
others = df[df.key != "deepseek-7b-base"]
checks.value("MAE identity holds for the other 16 models", "§5.4",
             (others.mae - others.mae_from_rule).abs().max(), 0, 1e-9)
g41 = df.set_index("name").loc["Granite~4.1 8B"]
opt = df.set_index("name").loc["OPT-6.7B"]
checks.same("model with the largest MAE", "§5.4", df.loc[df.mae.idxmax(), "name"], "Granite~4.1 8B")
checks.printed("MAE on Granite 4.1 8B", "§5.4", g41.mae, "6.18", 2)
checks.value("distance between l* and l0 on Granite 4.1 8B", "§5.4", abs(g41.target - g41.l0), 7)
checks.printed("MAE on OPT-6.7B", "§5.4", opt.mae, "0.14", 2)
checks.value("distance between l* and l0 on OPT-6.7B (adjacent)", "§5.4", abs(opt.target - opt.l0), 1)
df[["name", "target", "l0", "detected", "exact_pct", "mae", "mae_from_rule"]]

## Summary of checks

In [ ]:
df.drop(columns=["detected"]).to_csv(OUT / "detection_base_models.csv", index=False)
checks.table().to_csv(OUT / "checks_02_detection.csv", index=False)
checks.assert_all()
checks.table()

In [ ]:
inputs_used()